# 05 · Convolutions and a small 2D U-Net

    **Audience:** first-year biology students  
    **Time:** about 95 minutes  
    **Prerequisites:** Tutorials 01–04 and basic PyTorch tensors  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- read the `(batch, channel, height, width)` convention
- explain convolution, pooling, upsampling and skip connections
- train and inspect a tiny 2D segmentation U-Net

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 1 · Why not classify each pixel independently?
Neighboring pixels describe edges and shapes. A **convolution** slides a small learnable filter over an image, allowing a network to recognize local patterns. A U-Net downsamples to collect context and then upsamples to recover detail. **Skip connections** transfer high-resolution features across the U shape.

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(5)
torch.set_num_threads(min(2,torch.get_num_threads()))
rng = np.random.default_rng(5)
example = torch.zeros(1,1,32,32)   # N,C,H,W
example[0,0,10:24,12:25] = 1
edge_filter = nn.Conv2d(1,1,kernel_size=3,padding=1,bias=False)
with torch.no_grad():
    edge_filter.weight[:] = torch.tensor([[[[-1.,-1.,-1.],[-1.,8.,-1.],[-1.,-1.,-1.]]]])
edge_map = edge_filter(example)
fig,axes=plt.subplots(1,2,figsize=(7,3))
axes[0].imshow(example[0,0],cmap='gray');axes[0].set_title('Input mask')
axes[1].imshow(edge_map.detach()[0,0],cmap='coolwarm');axes[1].set_title('Hand-set convolution')
plt.tight_layout();plt.show()
print('Tensor shape:',tuple(example.shape))

## 2 · Create a safe, local training dataset
Our 2D images contain randomly positioned ellipses on a noisy background. Every image has a paired **ground-truth** mask. We use fixed but different seeds for training and validation to avoid copying the same example.

In [ ]:
def make_2d(n, seed, size=32):
    local_rng=np.random.default_rng(seed)
    yy,xx=np.mgrid[:size,:size]
    xs,ys=[],[]
    for _ in range(n):
        cy,cx=local_rng.integers(9,size-9,size=2)
        ry,rx=local_rng.integers(4,9,size=2)
        mask=(((yy-cy)/ry)**2+((xx-cx)/rx)**2<=1).astype(np.float32)
        img=np.clip(0.12+0.8*mask+local_rng.normal(0,0.10,(size,size)),0,1).astype(np.float32)
        xs.append(img[None]);ys.append(mask[None])
    return torch.tensor(np.stack(xs)),torch.tensor(np.stack(ys))
X_train,Y_train=make_2d(48,20)
X_val,Y_val=make_2d(8,21)
assert X_train.shape==(48,1,32,32)
fig,axes=plt.subplots(1,2,figsize=(6,3))
axes[0].imshow(X_train[0,0],cmap='gray');axes[0].set_title('Image')
axes[1].imshow(Y_train[0,0],cmap='gray');axes[1].set_title('True mask')
plt.show()

## 3 · Define a two-resolution U-Net
`Conv2d` extracts local features. `MaxPool2d` halves width/height. `ConvTranspose2d` doubles them. `torch.cat` creates a **skip connection** that combines encoder and decoder features. This compact network is deliberately smaller than research U-Nets.

In [ ]:
class TinyUNet2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1,4,3,padding=1),nn.ReLU(),nn.Conv2d(4,4,3,padding=1),nn.ReLU())
        self.down=nn.MaxPool2d(2)
        self.middle=nn.Sequential(nn.Conv2d(4,8,3,padding=1),nn.ReLU(),nn.Conv2d(8,8,3,padding=1),nn.ReLU())
        self.up=nn.ConvTranspose2d(8,4,2,stride=2)
        self.dec=nn.Sequential(nn.Conv2d(8,4,3,padding=1),nn.ReLU(),nn.Conv2d(4,4,3,padding=1),nn.ReLU())
        self.out=nn.Conv2d(4,1,1)
    def forward(self,x):
        high=self.enc(x)
        low=self.middle(self.down(high))
        up=self.up(low)
        merged=torch.cat([up,high],dim=1)  # skip connection
        return self.out(self.dec(merged)) # raw logits, not probabilities

model=TinyUNet2D()
with torch.no_grad():
    print('Input → output:',tuple(X_train[:1].shape),'→',tuple(model(X_train[:1]).shape))
assert model(X_train[:1]).shape == Y_train[:1].shape

## 4 · Train with a segmentation-specific loss
Foreground takes up a minority of pixels. We combine binary cross-entropy with soft Dice loss; logits are converted using `sigmoid` only for the Dice calculation. We use a few short epochs so the notebook runs on a CPU.

In [ ]:
def soft_dice_loss(logits,truth):
    probs=torch.sigmoid(logits)
    dims=(1,2,3)
    score=(2*(probs*truth).sum(dims)+1)/(probs.sum(dims)+truth.sum(dims)+1)
    return 1-score.mean()

def seg_loss(logits,truth):
    return F.binary_cross_entropy_with_logits(logits,truth)+soft_dice_loss(logits,truth)

optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
train_losses=[]
for epoch in range(12):
    model.train()
    for indices in torch.randperm(len(X_train)).split(8):
        optimizer.zero_grad()
        loss=seg_loss(model(X_train[indices]),Y_train[indices])
        loss.backward();optimizer.step()
    train_losses.append(float(loss.item()))
print('Final minibatch loss:',round(train_losses[-1],3))

## 5 · Look at a held-out prediction
Dice on the **validation** images is separate from the training loss. We show one image, its reference annotation, and the thresholded model probability.

In [ ]:
def binary_dice(pred,truth):
    pred=pred.bool();truth=truth.bool()
    total=(pred.sum()+truth.sum()).item()
    return 1.0 if total==0 else (2*(pred & truth).sum()/total).item()
model.eval()
with torch.no_grad():
    prob=torch.sigmoid(model(X_val))
pred=prob>0.5
score=np.mean([binary_dice(pred[i],Y_val[i]>0.5) for i in range(len(X_val))])
print('Mean validation Dice:',round(float(score),3))
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,im,title in zip(axes,[X_val[0,0],Y_val[0,0],prob[0,0]],['Image','Reference mask','Predicted probability']):
    ax.imshow(im,cmap='gray',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.tight_layout();plt.show()

## Your turn
1. Change the input image size from 32 to 40 (both divisible by 2) and inspect tensor sizes.
2. Print the shapes of `high`, `low`, and `up` in `forward`. Which is smallest?
3. Temporarily change `torch.cat([up, high], dim=1)` to `torch.cat([up, torch.zeros_like(high)], dim=1)` and retrain a **new** model. Does the skip connection help?

**Self-check:** The model output is a **logit** tensor with the same spatial shape as the label, not a class ID for the entire image.

In [ ]:
assert prob.shape==Y_val.shape
assert 0.0<=score<=1.0
assert all(np.isfinite(train_losses))
print('Shape and numeric checks passed.')

## Next
The same architecture extends to volumetric images if you replace 2D operations with 3D operations. But first: what is a voxel, and how is a Z-stack different from an ordinary picture?